In [ ]:
# Cell 1: Setup and Imports
baseDir = '/Users/shrey/NeuroSpeech/speechBCI-main'


import os
from glob import glob
from pathlib import Path
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]=""

import numpy as np
from omegaconf import OmegaConf
import tensorflow as tf
from neuralDecoder.neuralSequenceDecoder import NeuralSequenceDecoder
import neuralDecoder.utils.lmDecoderUtils as lmDecoderUtils

In [ ]:
# Cell 2: Load Language Model
# Load the language model, could take a while and requires ~60 GB of memory
print("Loading language model...")
lmDir = baseDir+'/languageModel'
ngramDecoder = lmDecoderUtils.build_lm_decoder(
    lmDir,
    acoustic_scale=0.8,
    nbest=1,
    beam=18
)
print("Language model loaded successfully")

In [ ]:
# Cell 3: Initialize Storage for Results
# Evaluate the HM-RNN on the test partition and competitionHoldOut partition
testDirs = ['test','competitionHoldOut']
trueTranscriptions = [[],[]]
decodedTranscriptions = [[],[]]
layerOutputs = [[],[]]  # Store intermediate layer outputs for analysis

In [ ]:
# Cell 4: Run Inference Loop
for dirIdx in range(2):
    print(f"\nEvaluating on {testDirs[dirIdx]} partition...")

    # CHANGED: Load HM-RNN checkpoint instead of baseline
    ckptDir = baseDir + '/derived/rnns/hmrnnRelease'

    args = OmegaConf.load(os.path.join(ckptDir, 'args.yaml'))
    args['loadDir'] = ckptDir
    args['mode'] = 'infer'
    args['loadCheckpointIdx'] = None

    for x in range(len(args['dataset']['datasetProbabilityVal'])):
        args['dataset']['datasetProbabilityVal'][x] = 0.0

    for sessIdx in range(4,19):
        args['dataset']['datasetProbabilityVal'][sessIdx] = 1.0
        args['dataset']['dataDir'][sessIdx] = baseDir+'/derived/tfRecords'
    args['testDir'] = testDirs[dirIdx]

    # Initialize model
    tf.compat.v1.reset_default_graph()
    nsd = NeuralSequenceDecoder(args)

    # Inference
    print("Running neural network inference...")
    out = nsd.inference()

    print("Applying language model decoding...")
    decoder_out = lmDecoderUtils.cer_with_lm_decoder(ngramDecoder, out, outputType='speech_sil', blankPenalty=np.log(2))

    def _ascii_to_text(text):
        endIdx = np.argwhere(text==0)
        return ''.join([chr(char) for char in text[0:endIdx[0,0]]])

    for x in range(out['transcriptions'].shape[0]):
        trueTranscriptions[dirIdx].append(_ascii_to_text(out['transcriptions'][x,:]))
    decodedTranscriptions[dirIdx] = decoder_out['decoded_transcripts']

    # Store intermediate layer outputs if available (for multi-scale analysis)
    if 'layer_outputs' in out:
        layerOutputs[dirIdx] = out['layer_outputs']

    print(f"Completed {testDirs[dirIdx]} partition")

In [ ]:
# Cell 4: Run Inference Loop
for dirIdx in range(2):
    print(f"\nEvaluating on {testDirs[dirIdx]} partition...")

    # CHANGED: Load HM-RNN checkpoint instead of baseline
    ckptDir = baseDir + '/derived/rnns/hmrnnRelease'

    args = OmegaConf.load(os.path.join(ckptDir, 'args.yaml'))
    args['loadDir'] = ckptDir
    args['mode'] = 'infer'
    args['loadCheckpointIdx'] = None

    for x in range(len(args['dataset']['datasetProbabilityVal'])):
        args['dataset']['datasetProbabilityVal'][x] = 0.0

    for sessIdx in range(4,19):
        args['dataset']['datasetProbabilityVal'][sessIdx] = 1.0
        args['dataset']['dataDir'][sessIdx] = baseDir+'/derived/tfRecords'
    args['testDir'] = testDirs[dirIdx]

    # Initialize model
    tf.compat.v1.reset_default_graph()
    nsd = NeuralSequenceDecoder(args)

    # Inference
    print("Running neural network inference...")
    out = nsd.inference()

    print("Applying language model decoding...")
    decoder_out = lmDecoderUtils.cer_with_lm_decoder(ngramDecoder, out, outputType='speech_sil', blankPenalty=np.log(2))

    def _ascii_to_text(text):
        endIdx = np.argwhere(text==0)
        return ''.join([chr(char) for char in text[0:endIdx[0,0]]])

    for x in range(out['transcriptions'].shape[0]):
        trueTranscriptions[dirIdx].append(_ascii_to_text(out['transcriptions'][x,:]))
    decodedTranscriptions[dirIdx] = decoder_out['decoded_transcripts']

    # Store intermediate layer outputs if available (for multi-scale analysis)
    if 'layer_outputs' in out:
        layerOutputs[dirIdx] = out['layer_outputs']

    print(f"Completed {testDirs[dirIdx]} partition")

In [ ]:
# Cell 5: Calculate Performance Metrics
from neuralDecoder.utils.lmDecoderUtils import _cer_and_wer as cer_and_wer

print("\n" + "="*60)
print("PERFORMANCE METRICS")
print("="*60)

# Get word error rate and phoneme error rate for the test set (cer is actually phoneme error rate here)
cer, wer = cer_and_wer(decodedTranscriptions[0], trueTranscriptions[0], outputType='speech_sil', returnCI=True)

# Print word error rate
print("\nTest Set Performance:")
print(f"  Word Error Rate (WER): {wer}")
print(f"  Character Error Rate (CER): {cer}")

In [ ]:
# Cell 6: Display Test Set Predictions
print("\nTest Set Decoded Transcriptions:")
print(decodedTranscriptions[0])

In [ ]:
# Cell 7: Display Competition Hold-Out Predictions
print("\nCompetition Hold-Out Decoded Transcriptions:")
print(decodedTranscriptions[1])

In [ ]:
# Cell 8: Optional Multi-Scale Analysis
# Optional: Analyze multi-scale behavior
if layerOutputs[0]:
    print("\n" + "="*60)
    print("MULTI-SCALE ANALYSIS")
    print("="*60)
    print("Layer outputs available for detailed temporal analysis")
    print("You can analyze phoneme-level (fast), syllable-level (medium),")
    print("and word-level (slow) representations")

In [ ]:
# Cell 9: Generate Competition Submission File
# Format the predictions for competition submission. This generates a .txt file that can be submitted.
print("\n" + "="*60)
print("GENERATING COMPETITION SUBMISSION")
print("="*60)

with open('hmrnnCompetitionSubmission.txt', 'w') as f:
    for x in range(len(decodedTranscriptions[1])):
        f.write(decodedTranscriptions[1][x]+'\n')

print(f"Competition submission saved to: hmrnnCompetitionSubmission.txt")
print(f"Total predictions: {len(decodedTranscriptions[1])}")